In [7]:
import torch
from torchvision.datasets import OxfordIIITPet
from torchvision import transforms
from torch.utils.data import DataLoader
import wandb

In [8]:
import torchvision.models as models
from torchvision.models import ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)
transform = weights.transforms()


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\shehr/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:21<00:00, 4.76MB/s]


In [9]:
dataset = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=transform,
    download=True,
)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

images, labels = next(iter(loader))
print(images.shape)   # [32, 3, 224, 224]
print(labels.shape)   # [32]

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [10]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

Model 3: Progressive unfreezing

In [27]:
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="shehryar11w-private",
    # Set the wandb project where this run will be logged.
    project="petDataset",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 1e-3,
        "architecture": "ResNet - Progressive Unfreezing",
        "dataset": "OxfordIIITPet",
        "epochs": 15,
    },
)


wandb: Initializing weave.


In [29]:
import torch.nn as nn
import torch.nn.functional as F

weights = ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)
transform = weights.transforms()

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 37)

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [31]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.fc.parameters(),lr=1e-3)

In [32]:
for epoch in range(15):  # loop over the dataset multiple times

    #Progressive Unfreezing
    if epoch == 5: # Epoch 6: unfreeze layer4
        print("Unfreezing layer4")

        for param in model.layer4.parameters():
            param.requires_grad = True

        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=1e-4
        )
    if epoch == 10:  # Epoch 11: unfreeze everything
        print("Unfreezing entire network")

        for param in model.parameters():
            param.requires_grad = True

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=1e-5
        )


    ### Train
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        # log statistics
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f'[{epoch + 1}] loss: {epoch_loss:.4f}')

    ### Validate
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:

            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = 100 * correct / total

    print(f"Epoch {epoch+1}: Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.2f}%")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": epoch_loss,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
    })

print('Finished Training')

[1] loss: 2.0909
Epoch 1: Val Loss = 1.1009, Val Acc = 89.27%
[2] loss: 0.6937
Epoch 2: Val Loss = 0.6201, Val Acc = 89.54%
[3] loss: 0.3942
Epoch 3: Val Loss = 0.4572, Val Acc = 90.35%
[4] loss: 0.2713
Epoch 4: Val Loss = 0.3874, Val Acc = 91.98%
[5] loss: 0.2086
Epoch 5: Val Loss = 0.3511, Val Acc = 91.71%
Unfreezing layer4
[6] loss: 0.1126
Epoch 6: Val Loss = 0.2097, Val Acc = 92.93%
[7] loss: 0.0300
Epoch 7: Val Loss = 0.1979, Val Acc = 93.48%
[8] loss: 0.0149
Epoch 8: Val Loss = 0.1999, Val Acc = 92.80%
[9] loss: 0.0086
Epoch 9: Val Loss = 0.2057, Val Acc = 93.48%
[10] loss: 0.0060
Epoch 10: Val Loss = 0.1939, Val Acc = 93.89%
Unfreezing entire network
[11] loss: 0.0040
Epoch 11: Val Loss = 0.1879, Val Acc = 93.75%
[12] loss: 0.0026
Epoch 12: Val Loss = 0.1778, Val Acc = 94.16%
[13] loss: 0.0016
Epoch 13: Val Loss = 0.1809, Val Acc = 94.43%
[14] loss: 0.0019
Epoch 14: Val Loss = 0.1863, Val Acc = 94.57%
[15] loss: 0.0016
Epoch 15: Val Loss = 0.1833, Val Acc = 94.70%
Finished Train

In [33]:
wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▁▂▅▄▆▆▆▆▇▇▇███
val_loss,█▄▃▃▂▁▁▁▁▁▁▁▁▁▁
epoch,15
train_loss,0.00159
val_accuracy,94.70109
val_loss,0.1833
